# Campaign Insights Ingestion

Reads Azure AI Content Understanding JSON results for historic marketing campaign
videos from a **OneLake shortcut** (Files/cu-results) and flattens them into a
Delta table, `campaign_insights`, for Power BI reporting.

No credentials are configured in this notebook. The shortcut to Azure Blob
Storage authenticates with the **Fabric workspace identity** (Entra ID, no
SAS/keys). See `fabric/README.md` for one-time setup.

Run order:
1. Create the `cu-results` shortcut once (see README).
2. Run this notebook on a schedule (Fabric pipeline Notebook activity, or the
   notebook's own built-in scheduler) to pick up newly analyzed videos.


In [ ]:
# Path to the OneLake shortcut created against the cu-results Blob container.
# Update SHORTCUT_PATH if you name the shortcut or Lakehouse differently.
SHORTCUT_PATH = "Files/cu-results"
DELTA_TABLE_NAME = "campaign_insights"
INGESTED_FILES_TABLE = "campaign_insights_ingested_files"


## List result files not yet ingested

Tracks processed blob paths in a small Delta table so reruns only append new campaign results.

In [ ]:
import json as _json
from pyspark.sql import Row
from pyspark.sql.utils import AnalysisException

try:
    ingested_df = spark.read.table(INGESTED_FILES_TABLE)
    already_ingested = {row.file_path for row in ingested_df.select("file_path").collect()}
except AnalysisException:
    already_ingested = set()


def list_json_files_recursive(path):
    """Walk the shortcut tree (results are written one campaign subfolder
    deep, e.g. Files/cu-results/<campaign_name>/<id>.json) and return every
    .json file path. notebookutils.fs.ls() is not recursive on its own.
    """
    found = []
    for entry in notebookutils.fs.ls(path):
        if entry.isDir:
            found.extend(list_json_files_recursive(entry.path))
        elif entry.path.endswith(".json"):
            found.append(entry.path)
    return found


all_files = list_json_files_recursive(SHORTCUT_PATH)
new_files = [f for f in all_files if f not in already_ingested]

print(f"Found {len(all_files)} total result file(s); {len(new_files)} new file(s) to ingest.")


## Flatten Content Understanding results

Content Understanding returns one or more `contents[]` items per video (one item
per segment, or one item for the whole video when `enableSegment=false`). Each
item has a `fields` object whose values carry a `type` plus a matching
`value*` property (`valueString`, `valueNumber`, `valueArray`, `valueObject`,
`valueBoolean`, `valueDate`, ...).

The flattener below is schema-agnostic: it works for any Content Understanding
analyzer field schema (not just the current commercial-video analyzer fields),
so new fields added later in `infra/analyzers/commercial-video.json` show up as
new columns automatically after a Delta `mergeSchema` write.


In [ ]:
def extract_field_value(field):
    """Pull the scalar/array Python value out of a Content Understanding field object."""
    if not isinstance(field, dict):
        return field
    field_type = field.get("type")
    if field_type == "array" or "valueArray" in field:
        return [extract_field_value(item) for item in field.get("valueArray", [])]
    if field_type == "object" or "valueObject" in field:
        return {k: extract_field_value(v) for k, v in field.get("valueObject", {}).items()}
    for value_key in ("valueString", "valueNumber", "valueBoolean", "valueDate", "valueInteger"):
        if value_key in field:
            return field[value_key]
    return field.get("value")


def flatten_result(file_path, result):
    """Yield one flat dict per content item (segment or whole-video) in a CU result."""
    contents = result.get("contents") or result.get("result", {}).get("contents", [])
    campaign_name = file_path.rsplit("/", 2)[-2] if "/" in file_path else None
    for index, content in enumerate(contents):
        if not isinstance(content, dict):
            continue
        row = {
            "file_path": file_path,
            "campaign_name": campaign_name,
            "content_index": index,
            "start_time_ms": content.get("startTimeMs") or content.get("start_time_ms"),
            "end_time_ms": content.get("endTimeMs") or content.get("end_time_ms"),
            "markdown": content.get("markdown"),
        }
        fields = content.get("fields", {}) or {}
        for field_name, field_value in fields.items():
            value = extract_field_value(field_value)
            # Store list/dict fields as JSON strings so every row has a flat,
            # Delta-friendly schema; Power BI can still parse them if needed.
            if isinstance(value, (list, dict)):
                value = _json.dumps(value, ensure_ascii=False)
            row[field_name] = value
        yield row


In [ ]:
def to_local_path(abfss_path):
    """notebookutils.fs.ls() returns absolute abfss://...@onelake.../Files/... paths,
    not paths relative to the shortcut. Map that to the local filesystem mount for the
    attached default Lakehouse (/lakehouse/default/Files/...) so open() can read it.
    """
    marker = "/Files/"
    index = abfss_path.find(marker)
    if index == -1:
        # Already a relative path (e.g. "Files/cu-results/...").
        return f"/lakehouse/default/{abfss_path}"
    return f"/lakehouse/default/Files/{abfss_path[index + len(marker):]}"


rows = []
for file_path in new_files:
    # notebookutils.fs has no open(); the attached default Lakehouse (and
    # its Files/cu-results shortcut) is also exposed on the local
    # filesystem under /lakehouse/default, so plain Python open() works.
    with open(to_local_path(file_path), "r") as handle:
        result = _json.loads(handle.read())
    rows.extend(flatten_result(file_path, result))

print(f"Flattened {len(rows)} content/segment row(s) from {len(new_files)} new file(s).")


## Write to the campaign_insights Delta table

Appends new rows and evolves the schema automatically as analyzer fields change.

In [ ]:
if rows:
    import pandas as pd
    pdf = pd.DataFrame(rows)
    new_df = spark.createDataFrame(pdf.astype(object).where(pdf.notnull(), None))
    (
        new_df.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DELTA_TABLE_NAME)
    )

    ingested_rows = [Row(file_path=f) for f in new_files]
    ingested_new_df = spark.createDataFrame(ingested_rows)
    (
        ingested_new_df.write.format("delta")
        .mode("append")
        .saveAsTable(INGESTED_FILES_TABLE)
    )
    print(f"Wrote {len(rows)} row(s) to {DELTA_TABLE_NAME}; recorded {len(new_files)} ingested file(s).")
else:
    print("No new campaign result files to ingest.")


## Next steps

- Build a Power BI report or a Fabric semantic model on top of `campaign_insights`
  (brand/competitor mention counts, sentiment distribution, CTA presence rate,
  humor mechanism themes, by `campaign_name`).
- Schedule this notebook from a Fabric Data Pipeline (Notebook activity, e.g.
  daily) so historic campaign uploads are picked up automatically.
- When ready, build a second synthetic table of simulated impressions/clicks/
  conversions keyed by `file_path`/`campaign_name`, join it to `campaign_insights`,
  and correlate simulated effectiveness against extracted attributes such as
  `CallToAction`, `EmotionSentiment`, or `HumorMechanism`. See fabric/README.md.
